# Oracle basics

An **oracle** is a black-box unitary that encodes a Boolean function
$f:\{0,1\}^n \to \{0,1\}$. Two common forms:

- **Bit-flip oracle**: $|x\rangle|y\rangle \to |x\rangle|y \oplus f(x)\rangle$
- **Phase oracle**: $|x\rangle \to (-1)^{f(x)}|x\rangle$

Here $f(x) = x_0 \wedge x_1$ (AND gate), which marks only $|11\rangle$.
We build both oracle forms on 2 input qubits and verify the action.

In [ ]:
import qiskit as qk
import qiskit_aer as qka

## Bit-flip oracle

A Toffoli gate flips the target qubit when both inputs are $|1\rangle$:
$|x_0 x_1 y\rangle \to |x_0 x_1\; y \oplus (x_0 \wedge x_1)\rangle$.

In [ ]:
def bitflip_oracle():
    qc = qk.QuantumCircuit(3, name="bitflip oracle")
    qc.ccx(0, 1, 2)
    return qc

print(bitflip_oracle().draw())

Prepare inputs in $|+\rangle|+\rangle$ and run the oracle. Only $|11\rangle$ flips the target.

In [ ]:
qc = qk.QuantumCircuit(3, 3)
qc.h(0)
qc.h(1)
qc.compose(bitflip_oracle(), inplace=True)
qc.measure(range(3), range(3))
print(qc.draw())

sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc, sim), shots=4096).result().get_counts()
for bits, n in sorted(counts.items()):
    print(f"  |{bits}>  {n:4d}")

## Phase oracle

A controlled-$Z$ gate applies $(-1)^{f(x)}$ to the input register:
$|x\rangle \to (-1)^{x_0 \wedge x_1}|x\rangle$.

In [ ]:
def phase_oracle():
    qc = qk.QuantumCircuit(2, name="phase oracle")
    qc.cz(0, 1)
    return qc

print(phase_oracle().draw())

In [ ]:
qc2 = qk.QuantumCircuit(2)
qc2.h(0)
qc2.h(1)
qc2.compose(phase_oracle(), inplace=True)
sv = qk.quantum_info.Statevector.from_instruction(qc2)
for state in ["00", "01", "10", "11"]:
    amp = sv.data[int(state[::-1], 2)]
    sign = "+" if amp.real >= 0 else "-"
    print(f"  |{state}>  amp = {sign}{abs(amp):.4f}  p = {sv.probabilities_dict()[state]:.4f}")

## Phase kickback

When the ancilla starts in $|{-}\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$,
the bit-flip oracle triggers **phase kickback**: the phase appears on the
input register instead of the ancilla.

In [ ]:
def bitflip_oracle_with_ancilla():
    qc = qk.QuantumCircuit(3, name="bitflip+kickback")
    qc.x(2)
    qc.h(2)
    qc.ccx(0, 1, 2)
    return qc

qc3 = qk.QuantumCircuit(3)
qc3.h(0)
qc3.h(1)
qc3.compose(bitflip_oracle_with_ancilla(), inplace=True)
sv3 = qk.quantum_info.Statevector.from_instruction(qc3)
for state, p in sorted(sv3.probabilities_dict().items()):
    if p > 0.01:
        print(f"  |{state}>  p = {p:.4f}")

Only the computational basis states with $f(x)=1$ carry the minus sign.
The ancilla returns to $|{-}\rangle$ unchanged — the function value has
been *kicked back* into the input register's phase.